In [1]:
!python -m pip install -e ..

Obtaining file:///Users/dkoffical/Documents/GitHub/cs321m_project
  Installing build dependencies ... - \ | / - \ | done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... - \ done
  Preparing editable metadata (pyproject.toml) ... - \ done
  Using cached torch-2.2.2-cp310-none-macosx_10_9_x86_64.whl (150.8 MB)
  Using cached scipy-1.15.3-cp310-cp310-macosx_10_13_x86_64.whl (38.7 MB)
  Using cached scikit_learn-1.7.2-cp310-cp310-macosx_10_9_x86_64.whl (9.3 MB)
  Using cached pandas-2.3.3-cp310-cp310-macosx_10_9_x86_64.whl (11.6 MB)
  Using cached pyarrow-24.0.0.tar.gz (1.2 MB)
  Installing build dependencies ... - \ | / - \ | / - \ | / done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... - done
  Using cached pyro_ppl-1.9.1-py3-none-any.whl (755 kB)
  Using cached huggingface_hub-1.16.4-py3-none-any.whl (668 kB)
  Using cached seaborn-0.13.2-py

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch_measure.datasets import load
from torch_measure.models import Rasch, TwoPL, ThreePL
from torch_measure.data import ResponseMatrix
from torch_measure.viz import plot_icc, plot_response_heatmap, plot_information
import torch_measure
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

import warnings
warnings.filterwarnings("ignore")

Using device: cpu


### Load a response matrix

Choose one matrix preset by changing `KFACTOR_MATRIX`. Rows are models/subjects, columns are items, and values are the observed score/correctness used by the factor model.


In [3]:
import pandas as pd
import torch
from torch_measure.data import ResponseMatrix

MATRIX_PRESETS = {
    "mmlu_solver": {
        "prefix": "mmlu_pro_solver",
        "dir": "../benchmarks/mmlu/response_matrices",
        "benchmark_id": "mmlu_pro_solver",
        "item_content_field": "source",
        "item_id_field": "pair_id",
        "value": "correct: 1.0=true, 0.0=false, NaN=unparsed/missing",
    },
    "mmlu_judging": {
        "prefix": "mmlu_pro_judging",
        "dir": "../benchmarks/mmlu/response_matrices",
        "benchmark_id": "mmlu_pro_judging",
        "item_content_field": "source",
        "item_id_field": "pair_id/order",
        "value": "judge score/correctness; NaN=missing",
    },
    "safety_all_attacks": {
        "prefix": "safety_all_attacks",
        "dir": "../benchmarks/safety/final_solver/response_matrices",
        "benchmark_id": "safety_all_attacks",
        "item_content_field": "source",
        "item_id_field": "attack_family:input_index",
        "value": "score: 1.0=harmful/successful, 0.0=not harmful/unsuccessful, NaN=missing",
    },
    "code_solver": {
        "prefix": "livecodebench",
        "dir": "../benchmarks/code/response_matrices",
        "benchmark_id": "livecodebench",
        "item_content_field": "difficulty",
        "item_id_field": "question_id",
        "value": "correct: 1.0=passes public tests, 0.0=fails public tests, NaN=missing",
    },
    "code_judge": {
        "prefix": "codejudgebench_pairwise",
        "dir": "../benchmarks/code/response_matrices",
        "benchmark_id": "codejudgebench_pairwise",
        "item_content_field": "difficulty",
        "item_id_field": "split:question_id:pair_index:ordering",
        "value": "correct: 1.0=selected correct solution, 0.0=selected incorrect solution, NaN=unparsed",
    },
    "harmmetric_eval": {
        "prefix": "harmmetric_eval",
        "dir": "../benchmarks/HarmMetric_Eval/response_matrices",
        "benchmark_id": "harmmetric_eval",
        "item_content_field": "source",
        "item_id_field": "prompt_id",
        "value": "overall_effectiveness_score: soft/fractional score in [0, 1]",
    },
    "kudge_challenge": {
        "prefix": "kudge_challenge_easy_hard",
        "dir": "../benchmarks/kudge/results/kudge_challenge_easy_hard/response_matrices",
        "benchmark_id": "kudge_challenge_easy_hard",
        "item_content_field": "prompt",
        "item_id_field": "id",
        "value": "correct: 1.0=true, 0.0=false",
        "enrich_kudge": True,
    },
    "kudge_judge": {
        "prefix": "kudge_judge_easy_hard",
        "dir": "../benchmarks/kudge/results/kudge_judge_easy_hard/response_matrices",
        "benchmark_id": "kudge_judge_easy_hard",
        "item_content_field": "prompt",
        "item_id_field": "id",
        "value": "correct: 1.0=true, 0.0=false",
        "enrich_kudge": True,
    },
}

KFACTOR_MATRIX = "code_judge"  # set by run_all_kfactor_notebooks.py
config = MATRIX_PRESETS[KFACTOR_MATRIX]
matrix_dir = config["dir"]
prefix = config["prefix"]

matrix_path = f"{matrix_dir}/{prefix}_response_matrix.csv"
subject_meta_path = f"{matrix_dir}/{prefix}_subject_metadata.csv"
item_meta_path = f"{matrix_dir}/{prefix}_item_metadata.csv"

print(f"Loading {KFACTOR_MATRIX}")
print(matrix_path)

df = pd.read_csv(matrix_path, index_col=0)
subject_meta = pd.read_csv(subject_meta_path)
item_meta = pd.read_csv(item_meta_path)

def enrich_kudge_item_metadata(item_meta):
    """Add prompt/source text from the original KUDGE dataset when available."""
    try:
        from datasets import load_dataset
    except ImportError:
        print("datasets is not installed; item metadata will only include response-matrix fields.")
        return item_meta

    try:
        ds = load_dataset("amphora/kudge-challenge", split="train")
    except Exception as exc:
        print(f"Could not load amphora/kudge-challenge; item metadata will only include response-matrix fields: {exc}")
        return item_meta

    source_rows = []
    for row in ds:
        if row.get("subset") not in {"Korean-Easy", "Korean-Hard"}:
            continue
        source_rows.append({
            "item_id": str(row.get("id")),
            "prompt": row.get("prompt"),
            "chosen": row.get("chosen"),
            "rejected": row.get("rejected"),
        })

    source_meta = pd.DataFrame(source_rows).drop_duplicates("item_id")
    enriched = item_meta.copy()
    enriched["item_id"] = enriched["item_id"].astype(str)
    enriched = enriched.merge(source_meta, on="item_id", how="left")
    print(f"Enriched item metadata with prompt text for {enriched['prompt'].notna().sum()}/{len(enriched)} items")
    return enriched

if config.get("enrich_kudge"):
    item_meta = enrich_kudge_item_metadata(item_meta)

item_content_field = config.get("item_content_field")
if item_content_field in item_meta.columns:
    item_contents = list(item_meta[item_content_field].fillna("").astype(str))
else:
    item_contents = list(item_meta.iloc[:, 0].astype(str))

rm = ResponseMatrix(
    data=torch.tensor(df.values, dtype=torch.float32),
    subject_ids=list(df.index.astype(str)),
    item_ids=list(df.columns.astype(str)),
    item_contents=item_contents,
    subject_metadata=subject_meta.to_dict("records"),
    info={
        "benchmark_id": config["benchmark_id"],
        "item_id_field": config["item_id_field"],
        "value": config["value"],
    },
)

print(f"{rm.n_subjects} models x {rm.n_items} items, density = {rm.density:.1%}")


Loading code_judge
../benchmarks/code/response_matrices/codejudgebench_pairwise_response_matrix.csv
9 models x 2103 items, density = 99.5%


### K-Factor Fit

Fit `LogisticFM` with `K=1` and `K=2` on the response matrix. Rows stay as models/subjects and columns stay as items. Here `Z` is item easiness, so item difficulty is `-Z`.


In [4]:
import json
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

from torch_measure.models import LogisticFM, predict_dense
from torch_measure.models.rotation import varimax_rotation
from torch_measure.metrics.functional import compute_all

# Shared dense matrix + observed mask for all K-factor fits.
data = rm.data.to(device).float()
mask = ~torch.isnan(data) & (data != -1)

print(f"K-factor data: {data.shape[0]} models x {data.shape[1]} items, observed={mask.float().mean().item():.1%}")


K-factor data: 9 models x 2103 items, observed=99.5%


In [5]:
def make_heldout_split(mask, test_frac=0.2, seed=123):
    """Split observed response entries into train/eval masks."""
    observed = mask.nonzero(as_tuple=False)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    perm = torch.randperm(observed.shape[0], generator=gen)
    n_eval = max(1, int(round(test_frac * observed.shape[0])))
    eval_idx = observed[perm[:n_eval]]

    train_mask = mask.clone()
    eval_mask = torch.zeros_like(mask, dtype=torch.bool)
    train_mask[eval_idx[:, 0], eval_idx[:, 1]] = False
    eval_mask[eval_idx[:, 0], eval_idx[:, 1]] = True
    return train_mask, eval_mask


def fit_logistic_fm_k(data, train_mask, k, device="cpu", max_epochs=1000, lr=0.03, seed=123, eval_mask=None, verbose=True):
    torch.manual_seed(seed + k)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed + k)

    model = LogisticFM(
        n_subjects=data.shape[0],
        n_items=data.shape[1],
        n_factors=k,
        device=device,
    )

    history = model.fit(
        data,
        mask=train_mask,
        method="mle",
        max_epochs=max_epochs,
        lr=lr,
        verbose=verbose,
    )

    with torch.no_grad():
        probs = predict_dense(model).detach()

    train_metrics = compute_all(probs, data, mask=train_mask)
    eval_metrics = compute_all(probs, data, mask=eval_mask) if eval_mask is not None else None

    return {
        "model": model,
        "history": history,
        "metrics": eval_metrics if eval_metrics is not None else train_metrics,
        "train_metrics": train_metrics,
        "eval_metrics": eval_metrics,
    }


kfactor_specs = {
    1: {"max_epochs": 1000, "lr": 0.03},
    2: {"max_epochs": 1000, "lr": 0.02},
}

HELDOUT_REPEATS = 1
HELDOUT_TEST_FRAC = 0.2

heldout_rows = []
for k, spec in kfactor_specs.items():
    for repeat in range(HELDOUT_REPEATS):
        split_seed = 123 + 1000 * repeat + k
        train_mask, eval_mask = make_heldout_split(mask, test_frac=HELDOUT_TEST_FRAC, seed=split_seed)
        print(f"\nHeld-out fit LogisticFM K={k}, repeat={repeat + 1}/{HELDOUT_REPEATS}")
        result = fit_logistic_fm_k(
            data,
            train_mask,
            k=k,
            device=device,
            max_epochs=spec["max_epochs"],
            lr=spec["lr"],
            seed=split_seed,
            eval_mask=eval_mask,
        )
        metrics = result["eval_metrics"]
        heldout_log_likelihood = metrics.get("log_likelihood")
        heldout_rows.append({
            "k": k,
            "repeat": repeat,
            "loss": -heldout_log_likelihood if heldout_log_likelihood is not None else None,
            "auc": metrics.get("auc"),
            "log_likelihood": heldout_log_likelihood,
            "brier": metrics.get("brier"),
            "ece": metrics.get("ece"),
        })

heldout_eval_raw = pd.DataFrame(heldout_rows)
heldout_eval_summary = (
    heldout_eval_raw
    .groupby("k", as_index=False)
    .agg(
        loss=("loss", "mean"),
        loss_std=("loss", "std"),
        auc=("auc", "mean"),
        auc_std=("auc", "std"),
        log_likelihood=("log_likelihood", "mean"),
        log_likelihood_std=("log_likelihood", "std"),
        brier=("brier", "mean"),
        brier_std=("brier", "std"),
        ece=("ece", "mean"),
        ece_std=("ece", "std"),
    )
)

# Final full-data fits are used for item difficulty estimates and exported tables.
kfactor_results = {}
for k, spec in kfactor_specs.items():
    print(f"\nFinal full-data fit LogisticFM K={k}")
    kfactor_results[k] = fit_logistic_fm_k(
        data,
        mask,
        k=k,
        device=device,
        max_epochs=spec["max_epochs"],
        lr=spec["lr"],
        seed=123,
    )

final_fit_summary = pd.DataFrame([
    {
        "k": k,
        "final_train_loss": result["history"]["losses"][-1],
        "final_train_auc": result["train_metrics"].get("auc"),
        "final_train_log_likelihood": result["train_metrics"].get("log_likelihood"),
        "final_train_brier": result["train_metrics"].get("brier"),
        "final_train_ece": result["train_metrics"].get("ece"),
    }
    for k, result in kfactor_results.items()
])

kfactor_fit_summary = heldout_eval_summary.merge(final_fit_summary, on="k", how="left")

display(kfactor_fit_summary.round(4))



Held-out fit LogisticFM K=1, repeat=1/1


MLE fitting: 100%|██████████| 1000/1000 [00:08<00:00, 121.29it/s, loss=0.160752]



Held-out fit LogisticFM K=2, repeat=1/1


MLE fitting: 100%|██████████| 1000/1000 [00:06<00:00, 156.92it/s, loss=0.084139]



Final full-data fit LogisticFM K=1


MLE fitting: 100%|██████████| 1000/1000 [00:05<00:00, 188.07it/s, loss=0.188003]



Final full-data fit LogisticFM K=2


MLE fitting: 100%|██████████| 1000/1000 [00:11<00:00, 85.46it/s, loss=0.101771]


,k,loss,loss_std,auc,auc_std,log_likelihood,log_likelihood_std,brier,brier_std,ece,ece_std,final_train_loss,final_train_auc,final_train_log_likelihood,final_train_brier,final_train_ece
0,1,1.0578,NaN,0.8069,NaN,-1.0578,NaN,0.1748,NaN,0.1302,NaN,0.1880,0.9349,-0.1880,0.0622,0.0252
1,2,1.4017,NaN,0.8137,NaN,-1.4017,NaN,0.1989,NaN,0.1742,NaN,0.1018,0.9757,-0.1018,0.0323,0.0159


### Item Difficulty With Laplace Uncertainty

For `LogisticFM`, `Z` is item easiness and `difficulty = -Z`. The uncertainty below uses a diagonal/conditional Laplace approximation: item factors and model factors are held fixed, and item-difficulty uncertainty is estimated from Bernoulli curvature `sum p(1-p)` over observed model responses.


In [6]:
def build_item_difficulty_table(result, rm, item_meta=None):
    model = result["model"]

    difficulty = model.difficulty.detach().cpu()
    difficulty_centered = difficulty - difficulty.mean()
    easiness_z = model.Z.detach().cpu()
    loadings = model.loadings.detach().cpu()

    # Rotate loadings for interpretability. For K=1 this is the identity.
    rotated_loadings, rotation = varimax_rotation(loadings)
    rotated_abilities = model.U.detach().cpu() @ rotation

    table = pd.DataFrame({
        "item_id": rm.item_ids,
        "difficulty": difficulty.numpy(),
        "difficulty_centered": difficulty_centered.numpy(),
        "easiness_z": easiness_z.numpy(),
    })

    for j in range(rotated_loadings.shape[1]):
        table[f"loading_factor_{j + 1}"] = rotated_loadings[:, j].numpy()

    loading_cols = [c for c in table.columns if c.startswith("loading_factor_")]
    table["dominant_factor"] = (
        table[loading_cols]
        .abs()
        .idxmax(axis=1)
        .str.replace("loading_factor_", "factor_", regex=False)
    )

    if item_meta is not None:
        table = table.merge(
            item_meta.reset_index(drop=True),
            left_index=True,
            right_index=True,
            how="left",
            suffixes=("", "_meta"),
        )

    model_factors = pd.DataFrame({"model": rm.subject_ids})
    for j in range(rotated_abilities.shape[1]):
        model_factors[f"factor_{j + 1}"] = rotated_abilities[:, j].numpy()

    return table, model_factors


item_difficulty_tables = {}
model_factor_tables = {}
for k, result in kfactor_results.items():
    item_table, model_table = build_item_difficulty_table(result, rm, item_meta=item_meta)
    item_difficulty_tables[k] = item_table
    model_factor_tables[k] = model_table

print("K=1 hardest items")
display(item_difficulty_tables[1].sort_values("difficulty_centered", ascending=False).head(20).round(3))

print("K=2 hardest items")
display(item_difficulty_tables[2].sort_values("difficulty_centered", ascending=False).head(20).round(3))


K=1 hardest items


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,dominant_factor,item_id_meta,selected_order,pair_id,question_id,split,ordering,difficulty_meta,gold
1248,gemini_2.5_flash_lite:abc338_c:237:bwd,18.850,20.486,-18.850,8.446,factor_1,gemini_2.5_flash_lite:abc338_c:237:bwd,1248,gemini_2.5_flash_lite:abc338_c:237,abc338_c,gemini_2.5_flash_lite,bwd,hard,B
236,claude_3.7_sonnet:3558:236:bwd,14.315,15.951,-14.315,8.241,factor_1,claude_3.7_sonnet:3558:236:bwd,236,claude_3.7_sonnet:3558:236,3558,claude_3.7_sonnet,bwd,medium,B
48,claude_3.7_sonnet:3453:48:bwd,14.263,15.898,-14.263,6.429,factor_1,claude_3.7_sonnet:3453:48:bwd,48,claude_3.7_sonnet:3453:48,3453,claude_3.7_sonnet,bwd,hard,B
1111,gemini_2.5_flash_lite:3535:100:bwd,13.686,15.322,-13.686,6.151,factor_1,gemini_2.5_flash_lite:3535:100:bwd,1111,gemini_2.5_flash_lite:3535:100,3535,gemini_2.5_flash_lite,bwd,medium,B
1433,qwen3_235b:3406:33:bwd,13.061,14.697,-13.061,2.852,factor_1,qwen3_235b:3406:33:bwd,1433,qwen3_235b:3406:33,3406,qwen3_235b,bwd,hard,B
1234,gemini_2.5_flash_lite:abc328_e:223:bwd,13.053,14.689,-13.053,7.492,factor_1,gemini_2.5_flash_lite:abc328_e:223:bwd,1234,gemini_2.5_flash_lite:abc328_e:223,abc328_e,gemini_2.5_flash_lite,bwd,hard,B
1235,gemini_2.5_flash_lite:abc330_e:224:bwd,12.411,14.047,-12.411,5.556,factor_1,gemini_2.5_flash_lite:abc330_e:224:bwd,1235,gemini_2.5_flash_lite:abc330_e:224,abc330_e,gemini_2.5_flash_lite,bwd,hard,B
593,gemini_2.5_flash:2920:12:bwd,12.043,13.679,-12.043,6.940,factor_1,gemini_2.5_flash:2920:12:bwd,593,gemini_2.5_flash:2920:12,2920,gemini_2.5_flash,bwd,medium,B
1358,gemini_2.5_flash_lite:abc385_d:347:bwd,11.893,13.529,-11.893,5.358,factor_1,gemini_2.5_flash_lite:abc385_d:347:bwd,1358,gemini_2.5_flash_lite:abc385_d:347,abc385_d,gemini_2.5_flash_lite,bwd,hard,B
2062,claude_4_sonnet:abc384_f:244:bwd,11.405,13.040,-11.405,6.586,factor_1,claude_4_sonnet:abc384_f:244:bwd,2062,claude_4_sonnet:abc384_f:244,abc384_f,claude_4_sonnet,bwd,hard,B


K=2 hardest items


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,loading_factor_2,dominant_factor,item_id_meta,selected_order,pair_id,question_id,split,ordering,difficulty_meta,gold
1768,claude_4_opus:abc365_e:150:fwd,15.578,16.992001,-15.578,4.266,-1.268,factor_1,claude_4_opus:abc365_e:150:fwd,1768,claude_4_opus:abc365_e:150,abc365_e,claude_4_opus,fwd,medium,A
1037,gemini_2.5_flash_lite:3024:26:fwd,14.958,16.372000,-14.958,1.938,2.085,factor_2,gemini_2.5_flash_lite:3024:26:fwd,1037,gemini_2.5_flash_lite:3024:26,3024,gemini_2.5_flash_lite,fwd,hard,A
1039,gemini_2.5_flash_lite:3026:28:fwd,14.510,15.924000,-14.510,1.883,2.021,factor_2,gemini_2.5_flash_lite:3026:28:fwd,1039,gemini_2.5_flash_lite:3026:28,3026,gemini_2.5_flash_lite,fwd,medium,A
912,gemini_2.5_flash:3252:331:fwd,14.011,15.425000,-14.011,3.837,-1.130,factor_1,gemini_2.5_flash:3252:331:fwd,912,gemini_2.5_flash:3252:331,3252,gemini_2.5_flash,fwd,medium,A
79,claude_3.7_sonnet:3720:79:fwd,13.940,15.354000,-13.940,2.386,3.274,factor_2,claude_3.7_sonnet:3720:79:fwd,79,claude_3.7_sonnet:3720:79,3720,claude_3.7_sonnet,fwd,medium,A
1937,claude_4_sonnet:3777:119:fwd,13.116,14.530000,-13.116,3.590,-1.050,factor_1,claude_4_sonnet:3777:119:fwd,1937,claude_4_sonnet:3777:119,3777,claude_4_sonnet,fwd,hard,A
2101,claude_4_sonnet:arc195_b:283:fwd,12.267,13.681000,-12.267,2.105,3.210,factor_2,claude_4_sonnet:arc195_b:283:fwd,2101,claude_4_sonnet:arc195_b:283,arc195_b,claude_4_sonnet,fwd,hard,A
86,claude_3.7_sonnet:3770:86:fwd,11.841,13.255000,-11.841,2.046,3.239,factor_2,claude_3.7_sonnet:3770:86:fwd,86,claude_3.7_sonnet:3770:86,3770,claude_3.7_sonnet,fwd,medium,A
1935,claude_4_sonnet:3771:117:fwd,11.782,13.196000,-11.782,2.024,2.825,factor_2,claude_4_sonnet:3771:117:fwd,1935,claude_4_sonnet:3771:117,3771,claude_4_sonnet,fwd,hard,A
27,claude_3.7_sonnet:3233:27:fwd,11.647,13.061000,-11.647,2.005,3.104,factor_2,claude_3.7_sonnet:3233:27:fwd,27,claude_3.7_sonnet:3233:27,3233,claude_3.7_sonnet,fwd,hard,A


In [7]:
def item_difficulty_laplace_se(model, data, mask):
    """Conditional diagonal Laplace SE for LogisticFM item difficulty.

    This treats fitted model factors/loadings as fixed and estimates curvature
    only for each item intercept/difficulty. For difficulty = -Z, the sign
    does not change the curvature.
    """
    data = data.to(model.device).float()
    mask = mask.to(model.device)

    with torch.no_grad():
        probs = predict_dense(model).clamp(1e-7, 1 - 1e-7)
        info = torch.where(mask, probs * (1 - probs), torch.zeros_like(probs)).sum(dim=0)
        raw_se = 1 / torch.sqrt(info.clamp_min(1e-8))

        # Our main reported difficulty is mean-centered. Under a diagonal
        # approximation, propagate uncertainty through d_j - mean(d).
        n_items = raw_se.numel()
        raw_var = raw_se.pow(2)
        centered_var = ((1 - 1 / n_items) ** 2) * raw_var + (raw_var.sum() - raw_var) / (n_items ** 2)
        centered_se = torch.sqrt(centered_var.clamp_min(1e-12))

    return raw_se.detach().cpu(), centered_se.detach().cpu()


item_difficulty_with_uncertainty = {}
for k, result in kfactor_results.items():
    raw_se, centered_se = item_difficulty_laplace_se(result["model"], data, mask)

    table = item_difficulty_tables[k].copy()
    table["difficulty_laplace_se"] = raw_se.numpy()
    table["difficulty_centered_laplace_se"] = centered_se.numpy()
    table["difficulty_centered_laplace_lo"] = table["difficulty_centered"] - 1.96 * table["difficulty_centered_laplace_se"]
    table["difficulty_centered_laplace_hi"] = table["difficulty_centered"] + 1.96 * table["difficulty_centered_laplace_se"]
    item_difficulty_with_uncertainty[k] = table.sort_values("difficulty_centered", ascending=False).reset_index(drop=True)

print("K=1 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[1].round(3))

print("K=2 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[2].round(3))


K=1 item difficulties with Laplace uncertainty


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,dominant_factor,item_id_meta,selected_order,pair_id,question_id,split,ordering,difficulty_meta,gold,difficulty_laplace_se,difficulty_centered_laplace_se,difficulty_centered_laplace_lo,difficulty_centered_laplace_hi
0,gemini_2.5_flash_lite:abc338_c:237:bwd,18.850000,20.486000,-18.850000,8.446,factor_1,gemini_2.5_flash_lite:abc338_c:237:bwd,1248,gemini_2.5_flash_lite:abc338_c:237,abc338_c,gemini_2.5_flash_lite,bwd,hard,B,3.157,3.157,14.299000,26.673000
1,claude_3.7_sonnet:3558:236:bwd,14.315000,15.951000,-14.315000,8.241,factor_1,claude_3.7_sonnet:3558:236:bwd,236,claude_3.7_sonnet:3558:236,3558,claude_3.7_sonnet,bwd,medium,B,1.631,1.633,12.751000,19.150999
2,claude_3.7_sonnet:3453:48:bwd,14.263000,15.898000,-14.263000,6.429,factor_1,claude_3.7_sonnet:3453:48:bwd,48,claude_3.7_sonnet:3453:48,3453,claude_3.7_sonnet,bwd,hard,B,2.273,2.274,11.442000,20.355000
3,gemini_2.5_flash_lite:3535:100:bwd,13.686000,15.322000,-13.686000,6.151,factor_1,gemini_2.5_flash_lite:3535:100:bwd,1111,gemini_2.5_flash_lite:3535:100,3535,gemini_2.5_flash_lite,bwd,medium,B,2.183,2.183,11.043000,19.600000
4,qwen3_235b:3406:33:bwd,13.061000,14.697000,-13.061000,2.852,factor_1,qwen3_235b:3406:33:bwd,1433,qwen3_235b:3406:33,3406,qwen3_235b,bwd,hard,B,13.568,13.562,-11.884000,41.278000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2098,claude_4_sonnet:3493:71:fwd,-17.753000,-16.117001,17.753000,-8.001,factor_1,claude_4_sonnet:3493:71:fwd,1889,claude_4_sonnet:3493:71,3493,claude_4_sonnet,fwd,easy,A,2.923,2.923,-21.846001,-10.388000
2099,gemini_2.5_flash:abc390_e:271:fwd,-17.868999,-16.233999,17.868999,-10.255,factor_1,gemini_2.5_flash:abc390_e:271:fwd,852,gemini_2.5_flash:abc390_e:271,abc390_e,gemini_2.5_flash,fwd,easy,A,1.758,1.759,-19.681000,-12.786000
2100,claude_4_opus:abc380_e:168:fwd,-18.298000,-16.662001,18.298000,-8.247,factor_1,claude_4_opus:abc380_e:168:fwd,1786,claude_4_opus:abc380_e:168,abc380_e,claude_4_opus,fwd,easy,A,3.043,3.043,-22.625999,-10.698000
2101,gemini_2.5_flash_lite:arc183_c:382:fwd,-18.360001,-16.724001,18.360001,-8.221,factor_1,gemini_2.5_flash_lite:arc183_c:382:fwd,1393,gemini_2.5_flash_lite:arc183_c:382,arc183_c,gemini_2.5_flash_lite,fwd,easy,A,3.043,3.042,-22.688000,-10.761000


K=2 item difficulties with Laplace uncertainty


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,loading_factor_2,dominant_factor,item_id_meta,selected_order,pair_id,question_id,split,ordering,difficulty_meta,gold,difficulty_laplace_se,difficulty_centered_laplace_se,difficulty_centered_laplace_lo,difficulty_centered_laplace_hi
0,claude_4_opus:abc365_e:150:fwd,15.578,16.992001,-15.578,4.266,-1.268,factor_1,claude_4_opus:abc365_e:150:fwd,1768,claude_4_opus:abc365_e:150,abc365_e,claude_4_opus,fwd,medium,A,2.182,2.185,12.710000,21.274000
1,gemini_2.5_flash_lite:3024:26:fwd,14.958,16.372000,-14.958,1.938,2.085,factor_2,gemini_2.5_flash_lite:3024:26:fwd,1037,gemini_2.5_flash_lite:3024:26,3024,gemini_2.5_flash_lite,fwd,hard,A,1.895,1.898,12.653000,20.091999
2,gemini_2.5_flash_lite:3026:28:fwd,14.510,15.924000,-14.510,1.883,2.021,factor_2,gemini_2.5_flash_lite:3026:28:fwd,1039,gemini_2.5_flash_lite:3026:28,3026,gemini_2.5_flash_lite,fwd,medium,A,1.856,1.859,12.280000,19.569000
3,gemini_2.5_flash:3252:331:fwd,14.011,15.425000,-14.011,3.837,-1.130,factor_1,gemini_2.5_flash:3252:331:fwd,912,gemini_2.5_flash:3252:331,3252,gemini_2.5_flash,fwd,medium,A,1.966,1.969,11.565000,19.284000
4,claude_3.7_sonnet:3720:79:fwd,13.940,15.354000,-13.940,2.386,3.274,factor_2,claude_3.7_sonnet:3720:79:fwd,79,claude_3.7_sonnet:3720:79,3720,claude_3.7_sonnet,fwd,medium,A,5.003,5.002,5.550000,25.157000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2098,claude_4_opus:3621:54:bwd,-11.899,-10.485000,11.899,-2.685,0.996,factor_1,claude_4_opus:3621:54:bwd,1672,claude_4_opus:3621:54,3621,claude_4_opus,bwd,medium,B,1.438,1.443,-13.313000,-7.657000
2099,gemini_2.5_flash:3316:335:bwd,-11.943,-10.529000,11.943,-2.058,-3.240,factor_2,gemini_2.5_flash:3316:335:bwd,916,gemini_2.5_flash:3316:335,3316,gemini_2.5_flash,bwd,hard,B,4.286,4.285,-18.929001,-2.130000
2100,gemini_2.5_flash:abc328_e:165:fwd,-13.170,-11.756000,13.170,-3.607,1.056,factor_1,gemini_2.5_flash:abc328_e:165:fwd,746,gemini_2.5_flash:abc328_e:165,abc328_e,gemini_2.5_flash,fwd,easy,A,1.862,1.865,-15.411000,-8.101000
2101,claude_4_sonnet:abc390_f:252:bwd,-13.548,-12.134000,13.548,-3.700,1.082,factor_1,claude_4_sonnet:abc390_f:252:bwd,2070,claude_4_sonnet:abc390_f:252,abc390_f,claude_4_sonnet,bwd,easy,B,1.908,1.911,-15.879000,-8.389000


In [8]:
# Optional saves
result_name = KFACTOR_MATRIX
out_dir = Path("results") / result_name
out_dir.mkdir(parents=True, exist_ok=True)

def item_score_records(df, item_ids):
    """Return {item_id: {model: observed score}} with NaN converted to None."""
    score_map = {}
    for item_id in item_ids:
        values = df[str(item_id)] if str(item_id) in df.columns else df[item_id]
        score_map[str(item_id)] = {
            str(model): (None if pd.isna(value) else float(value))
            for model, value in values.items()
        }
    return score_map

scores_by_item = item_score_records(df, rm.item_ids)

kfactor_fit_summary.to_csv(out_dir / f"{result_name}_kfactor_fit_summary.csv", index=False)
item_difficulty_json = {
    "matrix": result_name,
    "benchmark_id": rm.info.get("benchmark_id"),
    "item_id_field": rm.info.get("item_id_field"),
    "value": rm.info.get("value"),
    "fits": {},
}
for k, table in item_difficulty_with_uncertainty.items():
    table.to_csv(out_dir / f"{result_name}_kfactor_k{k}_item_difficulties_with_laplace_uncertainty.csv", index=False)
    model_factor_tables[k].to_csv(out_dir / f"{result_name}_kfactor_k{k}_model_factors.csv", index=False)

    records = table.astype(object).where(pd.notna(table), None).to_dict("records")
    for record in records:
        record["scores"] = scores_by_item.get(str(record["item_id"]), {})
    item_difficulty_json["fits"][f"k{k}"] = records

with open(out_dir / f"{result_name}_kfactor_item_difficulties_with_laplace_uncertainty.json", "w") as f:
    json.dump(item_difficulty_json, f, indent=2)

print(f"Saved K-factor tables to {out_dir.resolve()}")


Saved K-factor tables to /Users/dkoffical/Documents/GitHub/cs321m_project/K-Factor/results/code_judge


In [9]:
item_difficulty_json['fits']

{'k1': [{'item_id': 'gemini_2.5_flash_lite:abc338_c:237:bwd',
   'difficulty': 18.850095748901367,
   'difficulty_centered': 20.48584747314453,
   'easiness_z': -18.850095748901367,
   'loading_factor_1': 8.446344375610352,
   'dominant_factor': 'factor_1',
   'item_id_meta': 'gemini_2.5_flash_lite:abc338_c:237:bwd',
   'selected_order': 1248,
   'pair_id': 'gemini_2.5_flash_lite:abc338_c:237',
   'question_id': 'abc338_c',
   'split': 'gemini_2.5_flash_lite',
   'ordering': 'bwd',
   'difficulty_meta': 'hard',
   'gold': 'B',
   'difficulty_laplace_se': 3.1569457054138184,
   'difficulty_centered_laplace_se': 3.156541109085083,
   'difficulty_centered_laplace_lo': 14.299026489257812,
   'difficulty_centered_laplace_hi': 26.67266845703125,
   'scores': {'claude_haiku_4_5': 0.0,
    'mistral_3b': 0.0,
    'mistral_8b': 0.0,
    'mistral_14b': 0.0,
    'qwen3_5_0_8b': 0.0,
    'qwen3_5_2b': 1.0,
    'qwen3_5_4b': 1.0,
    'qwen3_5_9b': 0.0,
    'qwen3_5_27b': 0.0}},
  {'item_id': 'claude